# 🐋 Orca - Data Exploration Notebook

Premier notebook pour explorer les données collectées par Orca Trading Bot.

In [2]:
# Cell 1: Setup
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Add project root to path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

print("🐋 Orca Data Exploration Notebook")
print(f"Project root: {project_root}")

🐋 Orca Data Exploration Notebook
Project root: c:\Users\User\orca\orca


In [3]:
# Cell 2: Initialize and show config
from orca.data.collector import DataCollector
from orca.core.config import config

collector = DataCollector()
config.show()

2026-01-05 17:08:46,730 - orca.core.config - INFO - Loaded config from c:\Users\User\orca\orca\config\default.yaml
2026-01-05 17:08:46,734 - orca.core.config - INFO - Configuration loaded successfully
2026-01-05 17:08:46,754 - orca.data.collector - INFO - Database initialized at c:\Users\User\orca\orca\data\processed\market.db
2026-01-05 17:08:46,756 - orca.data.collector - INFO - Initialized DataCollector for binance


🐋 ORCA TRADING CONFIGURATION

📊 Data:
  Symbols: ['BTC/USDT']
  Timeframes: ['5m', '15m', '1h']
  Exchange: binance
  Database: c:\Users\User\orca\orca\data\processed\market.db

💰 Trading:
  Initial balance: $10,000.00
  Commission: 0.1%
  Max position size: 10%

🤖 Model:
  Training episodes: 1000
  Validation split: 0.2

⚠️  Risk:
  Max drawdown: 10%
  Stop loss: 2%


In [4]:
# Cell 3: Check available data
print("\n📊 Checking available data...")
available_data = collector.get_available_data()

if available_data.empty:
    print("No data available. Run scripts/fetch_data.py first!")
else:
    print("\nAvailable data:")
    print(available_data.to_string(index=False))


📊 Checking available data...

Available data:
  symbol timeframe          first_date           last_date  candle_count  days_available
BTC/USDT       15m 2025-12-29 16:00:00 2026-01-05 15:45:00           672               6
BTC/USDT        1h 2025-12-29 16:00:00 2026-01-05 15:00:00           168               6
BTC/USDT        5m 2025-12-29 16:00:00 2026-01-05 15:55:00          2016               6


In [5]:
# Cell 4: Load and display sample data
if not available_data.empty:
    symbol = config.data.symbols[0]
    timeframe = config.data.timeframes[0]
    
    print(f"\n📈 Loading data for {symbol} {timeframe}...")
    
    data = collector.fetch_historical_data(symbol, timeframe, days=7)
    
    if not data.empty:
        print(f"\nData shape: {data.shape}")
        print(f"Date range: {data.index[0]} to {data.index[-1]}")
        
        # Display first and last few rows
        print("\nFirst 5 rows:")
        print(data.head())
        print("\nLast 5 rows:")
        print(data.tail())
    else:
        print(f"No data available for {symbol} {timeframe}")

2026-01-05 17:09:17,219 - orca.data.collector - INFO - Fetching data for BTC/USDT 5m (7 days)
2026-01-05 17:09:17,246 - orca.data.collector - ERROR - Error fetching data for BTC/USDT 5m: NaTType does not support timestamp



📈 Loading data for BTC/USDT 5m...
No data available for BTC/USDT 5m


In [6]:
# Cell 5: Basic statistics
if 'data' in locals() and not data.empty:
    print("\n📈 Basic Statistics:")
    print(data.describe())

In [ ]:
# Cell 6: Visualize price
if 'data' in locals() and not data.empty:
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=(f"{symbol} Price", "Volume"),
        row_heights=[0.7, 0.3]
    )
    
    # Price chart
    fig.add_trace(
        go.Candlestick(
            x=data.index,
            open=data['open'],
            high=data['high'],
            low=data['low'],
            close=data['close'],
            name='OHLC'
        ),
        row=1, col=1
    )
    
    # Volume chart
    fig.add_trace(
        go.Bar(
            x=data.index,
            y=data['volume'],
            name='Volume',
            marker_color='lightblue'
        ),
        row=2, col=1
    )
    
    fig.update_layout(
        title=f"{symbol} {timeframe} - Last 7 Days",
        yaxis_title="Price (USDT)",
        yaxis2_title="Volume",
        xaxis_rangeslider_visible=False,
        height=600,
        template="plotly_dark"
    )
    
    fig.show()

In [ ]:
# Cell 7: Data quality check
if 'data' in locals() and not data.empty:
    print("\n🔍 Data Quality Check:")
    total_candles = len(data)
    
    # Define timeframe to minutes mapping
    timeframe_map = {
        '1m': 1,
        '5m': 5,
        '15m': 15,
        '1h': 60,
        '4h': 240,
        '1d': 1440,
    }
    
    minutes_per_candle = timeframe_map.get(timeframe, 60)
    expected_candles = 7 * 24 * 60 / minutes_per_candle
    completeness = (total_candles / expected_candles) * 100
    
    print(f"Expected candles: {expected_candles:.0f}")
    print(f"Actual candles: {total_candles}")
    print(f"Completeness: {completeness:.1f}%")
    
    # Check for gaps
    time_diff = data.index.to_series().diff()
    avg_gap = time_diff.mean()
    max_gap = time_diff.max()
    
    print(f"\nAverage time between candles: {avg_gap}")
    print(f"Maximum gap: {max_gap}")

In [ ]:
# Cell 8: Calculate returns
if 'data' in locals() and not data.empty:
    data['returns'] = data['close'].pct_change()
    data['log_returns'] = np.log(data['close'] / data['close'].shift(1))
    
    print("\n📊 Returns Statistics:")
    print(f"Mean daily return: {data['returns'].mean() * 100:.4f}%")
    print(f"Return volatility: {data['returns'].std() * 100:.4f}%")
    
    if data['returns'].std() > 0:
        sharpe = data['returns'].mean() / data['returns'].std()
        print(f"Sharpe ratio (assuming 0% risk-free): {sharpe:.4f}")
    
    # Plot returns distribution
    fig_returns = px.histogram(
        data['returns'].dropna(),
        nbins=50,
        title="Distribution of Returns",
        labels={'value': 'Return', 'count': 'Frequency'},
        color_discrete_sequence=['lightgreen']
    )
    fig_returns.show()
    
    # Plot cumulative returns
    data['cumulative_return'] = (1 + data['returns']).cumprod() - 1
    
    fig_cumulative = go.Figure()
    fig_cumulative.add_trace(go.Scatter(
        x=data.index,
        y=data['cumulative_return'] * 100,
        mode='lines',
        name='Cumulative Return',
        line=dict(color='green', width=2)
    ))
    
    fig_cumulative.update_layout(
        title="Cumulative Returns",
        xaxis_title="Date",
        yaxis_title="Cumulative Return (%)",
        template="plotly_dark"
    )
    
    fig_cumulative.show()

In [ ]:
# Cell 9: Save analysis results
if 'data' in locals() and not data.empty:
    # Create analysis summary
    analysis_summary = {
        'symbol': symbol,
        'timeframe': timeframe,
        'start_date': data.index[0].strftime('%Y-%m-%d'),
        'end_date': data.index[-1].strftime('%Y-%m-%d'),
        'total_candles': len(data),
        'price_range': f"${data['low'].min():,.2f} - ${data['high'].max():,.2f}",
        'mean_return': f"{data['returns'].mean() * 100:.4f}%",
        'volatility': f"{data['returns'].std() * 100:.4f}%",
        'data_completeness': f"{completeness:.1f}%"
    }
    
    print("\n📋 Analysis Summary:")
    for key, value in analysis_summary.items():
        print(f"{key.replace('_', ' ').title()}: {value}")
    
    # Save to CSV
    import os
    os.makedirs('data/analysis', exist_ok=True)
    
    # Save raw data sample
    data_sample = data.copy()
    data_sample.reset_index(inplace=True)
    data_sample.to_csv(f'data/analysis/{symbol.replace("/", "_")}_{timeframe}_sample.csv', index=False)
    
    print(f"\n💾 Data sample saved to: data/analysis/{symbol.replace('/', '_')}_{timeframe}_sample.csv")

# 🎯 Next Steps

1. **Run backtesting**: Execute `python scripts/backtest_baseline.py`
2. **Collect more data**: Modify `scripts/fetch_data.py` to get more history
3. **Build features**: Add technical indicators in next notebook
4. **Train first model**: Start with simple RL agent